In [0]:
# @title library installation
!pip install openai==1.55.3

In [0]:
dbutils.library.restartPython()

In [0]:
# @title IMPORTING ALL THE NECESSARY LIBRARIES FOR PROJ
import openai
import time
import random
from openai import OpenAI
import pandas as pd
from pyspark.sql.functions import current_timestamp
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import httpx
import requests
from requests.auth import HTTPBasicAuth
import json
from datetime import datetime
import pytz
from collections import defaultdict
import hashlib
import threading
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

In [0]:
# OpenAI
openai.api_key = "enter_the_api_key_here"

In [0]:
#this is the dataframe that holds the comments for classification
main_df = pd.read_csv('enter_the_path_here')

In [0]:
main_df

In [0]:
#this is the dataframe that holds the test dataset for validation [should hold 'Comment', 'Module' columns]
module_test_df = pd.read_csv('enter_the_path_here')
module_test_df

In [0]:
spark_table_path = "/dbfs/FileStore/tables/feedback_classifier/batch_tracker/module_batch_tracker_test"

schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("status", StringType(), True),
    StructField("submitted_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("output_file_id", StringType(), True),
    StructField("description", StringType(), True),
    StructField("manual_indices_json", StringType(), True),              # NEW
    StructField("original_manual_comments_json", StringType(), True)    # NEW
])

In [0]:
def upload_batch_file(file_path):
    try:
        with open(file_path, "rb") as file:
            response = requests.post(
                "https://api.openai.com/v1/files",
                headers={"Authorization": f"Bearer {openai.api_key}"},
                files={"file": file},
                data={"purpose": "batch"}
            )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error uploading batch file: {e}")
        return None

def create_batch_job(input_file_id):
    try:
        response = requests.post(
            "https://api.openai.com/v1/batches",
            headers={
                "Authorization": f"Bearer {openai.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "input_file_id": input_file_id,
                "endpoint": "/v1/chat/completions",
                "completion_window": "24h",
                "metadata": {"description": "Processing review comments in batches"}
            }
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error creating batch job: {e}")
        return None

In [0]:
def batch_process_and_validate(module_test_df, comments, manual_comments, system_instructions, batch_size=50, injection_percentage=20, filename='batchinput.jsonl'):

    # ====== helper functions reused from your code ======
    def clean_response(response_text):
        lines = response_text.split("\n")
        cleaned_lines = [re.sub(r"^\d+\.\s*", "", line).rstrip() for line in lines]
        cleaned_response = "\n".join(cleaned_lines)
        return cleaned_response

    def calculate_manual_injection_count(batch_size, injection_percentage):
        return max(1, int((injection_percentage / 100) * batch_size))

    def calculate_frequency(batch_size, manual_count):
        if manual_count == 0:
            return batch_size
        return max(1, batch_size // manual_count)

    def inject_manual_data(batch_comments, manual_comments, manual_count, frequency):
        new_batch_comments = batch_comments.copy()
        manual_indices = []
        injected_manual_comments = []
        random.shuffle(manual_comments)
        manual_comments_to_inject = manual_comments[:manual_count]

        for i in range(manual_count):
            inject_index = (i + 1) * frequency - 1
            if inject_index < len(new_batch_comments):
                new_batch_comments.insert(inject_index, manual_comments_to_inject[i])
                manual_indices.append(inject_index)
                injected_manual_comments.append(manual_comments_to_inject[i])

        return new_batch_comments, manual_indices, injected_manual_comments

    def create_batch_input_file(comments, manual_comments, system_instructions, batch_size=30, injection_percentage=20, filename='batchinput.jsonl'):
        jsonl_entries = []
        all_manual_indices = []
        original_manual_comments = []
        current_index = 2

        with open(filename, 'w') as f:
            for idx, batch_start in enumerate(range(0, len(comments), batch_size)):
                batch_comments = comments[batch_start:batch_start + batch_size]
                no_of_injections = calculate_manual_injection_count(len(batch_comments), injection_percentage)
                frequency = calculate_frequency(len(batch_comments), no_of_injections)

                batch_comments_with_manual, manual_indices, injected_manuals = inject_manual_data(batch_comments, manual_comments, no_of_injections, frequency)

                all_manual_indices.append(manual_indices)
                original_manual_comments.append(injected_manuals)

                indexed_comments = '\n'.join([f"{current_index + i}. {comment}" for i, comment in enumerate(batch_comments_with_manual)])
                current_index += len(batch_comments_with_manual)

                request = {
                    "custom_id": f"request-{idx + 1}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": "gpt-4.1-2025-04-14",
                        "messages": [
                            {"role": "system", "content": system_instructions},
                            {"role": "user", "content": indexed_comments}
                        ],
                        "max_tokens": 1600
                    }
                }

                request_json = json.dumps(request)
                jsonl_entries.append(request_json)
                f.write(request_json + '\n')

        return all_manual_indices, original_manual_comments

    def extract_classified_comments(output_filename='batch_output.jsonl'):
        classified_comments = []
        try:
            with open(output_filename, 'r') as f:
                for line in f:
                    result = json.loads(line)
                    classified_comment = result['response']['body']['choices'][0]['message']['content']
                    cleaned_comment = clean_response(classified_comment)
                    classified_comments.append(cleaned_comment)
        except Exception as e:
            print(f"Error reading batch output file: {e}")
        return classified_comments

    # ====== main flow begins ======
    manual_data = dict(zip(module_test_df['Comment'], module_test_df['Module']))
    all_manual_indices, original_manual_comments = create_batch_input_file(comments, manual_comments, system_instructions, batch_size, injection_percentage, filename)
    print("Original Manual Comments:", original_manual_comments)
    
    # Upload and run batch job
    batch_input_file = upload_batch_file(filename)
    input_file_id = batch_input_file.get('id')
    print(f"Batch input file uploaded with ID: {input_file_id}")
    
    batch_job = create_batch_job(input_file_id)
    batch_id = batch_job.get('id')
    print(f"Batch job created with ID: {batch_id}")

    manual_indices_json = json.dumps(all_manual_indices)
    manual_comments_json = json.dumps(original_manual_comments)

    # Build row data
    tracker_data = [(
    batch_id,
    "submitted",
    None,  # submitted_at added below
    None,
    None,
    batch_job['metadata'].get('description', 'N/A'),
    manual_indices_json,
    manual_comments_json)]

    # Create DataFrame with schema
    df_tracker = spark.createDataFrame(tracker_data, schema)
    # Add submitted_at timestamp
    df_tracker = df_tracker.withColumn("submitted_at", current_timestamp())
    # Write to delta table
    df_tracker.write.format("delta").mode("append").save(spark_table_path)
    print(f"Tracker updated for batch_id: {batch_id}")

    return batch_id, all_manual_indices, original_manual_comments

In [0]:
system_instructions = """This GPT classifies feedback shared by users of a 'usecase (like ecom app, or trading app or any other usecase etc.)' into user journeys.

Task Overview:

Objective: Classify customer feedback into predefined modules based on initial knowledge.

Initial Knowledge on Classification (below is an example of the classification fields):
Classification fields are below:
1.Home: 'ENTER_THE_CONTEXT_FOR_USECASE'
2.Search: 'ENTER_THE_CONTEXT_FOR_USECASE'
3.Product: 'ENTER_THE_CONTEXT_FOR_USECASE'
4.Cart: 'ENTER_THE_CONTEXT_FOR_USECASE'
5.Checkout: 'ENTER_THE_CONTEXT_FOR_USECASE'
6.Payment: 'ENTER_THE_CONTEXT_FOR_USECASE'
7.Order: 'ENTER_THE_CONTEXT_FOR_USECASE'

Constraints and Requirements:
- Code Interpreter Usage: Limited to loading and writing CSV/XLS/XLSX files. No computational language detection or content analysis using code.
- Data Integrity: Ensure that each comment is analyzed as is, without any summarisation or modification. If a duplicate comment appears, it must not be removed or ignored
- Analysis Approach: Analyze comments individually, avoiding shortcuts or placeholders. Verify work to ensure human-like classification accuracy.

Special Notes:
- Comment should be classified based on the priority of the module.
- ENTER_THE_CONTEXT_FOR_USECASE

Output Requirements:
- You can give a maximum of 3 classification module per feedback/comment based on priority

- Output Table:
1 Publish the output with only the classified modules, excluding the header row, such that it can be copied directly into google sheets in each row. The output should be row by row that is in new lines so that it can be copied to gsheets rows. Headers should not be included in the output.

- Output restriction:
1 You can classify into multiple modules per comment if there are multiple reasons and it should be based on priority of the module. And don't exceed more than 3 module types per comment. Multiple modules can be separated by a comma.
2 You can refer all the ENTER_THE_COUNT classification types and do the classification accordingly and do not classify outside the (ENTER_THE_COUNT) mentioned types (if a comment is not matching to any of above mentioned modules, classify it to the closest mentioned module type from any of the mentioned list)

- Mandatory Fields in the output:
1 The response should be like this as mentioned in quotes - "index.Module"
2 Index and Module alone (Give only the classification for each input comment and don't give the description for the modules. I only need the type of module from the above classification field types that I provided you. Do it accurately by understanding all the module types. I need the module classification type alone in the response don't ever include sentiment types in the response)
3 Do as per the Index and return the same index as you see in the comment, if you see a comment that has for example 42 as index and then you should return the same index 42 in the classification output
4 The output format should not contain any headers or enclosed by special characters like ``` or double quotes. Only the index and classifications should be there. If you give headers or quotes then there will be an index mismatching for and parsing issues.
"""

In [0]:
batch_id, all_manual_indices, original_manual_comments = batch_process_and_validate(
    module_test_df=module_test_df,
    comments=main_df['comment'].dropna().tolist(),
    manual_comments=module_test_df['Comment'].dropna().tolist(),
    system_instructions=system_instructions
)

In [0]:
#storing on a dbr table
spark_table_path = "enter_the_table_path_here"
# Read Delta Table
status_df = spark.read.format("delta").load(spark_table_path)

print(f"Total batches found: {status_df.count()}")
status_df.show()